# Config 2: QLoRA fine-tune (GPU)

Trains a QLoRA adapter on top of `google/gemma-3-4b-it` -- the same base
model config1 serves via Ollama -- so that "config1 vs config2" in the
final results table isolates the effect of fine-tuning rather than
comparing two different models. This notebook is a runnable copy of
`SETUP.md` in this folder (section numbers match); the plan-level framing
(why config2 exists, what it's being compared against) is in the repo
root's `PLAN.md`.

**Hard constraint:** training data must never overlap `../eval-programs/`
(the 20 held-out programs used for scoring). `generate_training_corpus.py`
enforces this automatically below, on two independent signals.

Run top to bottom: get the repo -> get a GPU -> install deps -> generate
training data -> fine-tune (incl. Hugging Face auth) -> generate
predictions -> score base vs. tuned -> *(optional, section 6)* harvest
real SAS metadata over ODA and re-score against it.

## 0. Get the whole repo

Pull the **whole** repo, not just this folder: sections 3, 5 and 6 need
`../eval-programs/` (the 20 held-out programs + gold) and `../results/`
(`score.py`, `run_eval.py`) as siblings of this folder. Pulling only
`config2-qlora-gpu/` (e.g. via `npx degit .../config2-qlora-gpu`) leaves
those missing and every `../results/...` / `../eval-programs/...` call
below fails with "No such file or directory".

Off Colab this cell only reports where it is running, so the notebook is
the same file in both places.

In [1]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/patrickjlong1/sas-llm-paper.git"
CLONE_DIR = "/content/sas-llm-paper"

if IN_COLAB:
    if not os.path.isdir(CLONE_DIR):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, CLONE_DIR],
                       check=True)
    os.chdir(os.path.join(CLONE_DIR, "config2-qlora-gpu"))

print("running on Colab:", IN_COLAB)
print("cwd:            ", os.getcwd())
for need in ("../eval-programs/programs", "../eval-programs/gold", "../results"):
    print("  %-28s %s" % (need, "OK" if os.path.isdir(need) else "MISSING"))

running on Colab: True
cwd:             /content/sas-llm-paper/config2-qlora-gpu
  ../eval-programs/programs    OK
  ../eval-programs/gold        OK
  ../results                   OK


## 1. Get a free GPU

Colab UI action, not a cell to run: **Runtime -> Change runtime type ->
Hardware accelerator: T4 GPU -> Save**, before running anything below.
(Kaggle Notebooks and Lightning AI Studios work the same way if Colab's
free tier is unavailable -- see `SETUP.md` section 1.)

Everything from section 4 on needs CUDA: QLoRA's 4-bit quantization comes
from `bitsandbytes`, whose kernels are GPU-only. The check below fails
loudly now rather than letting you discover it after a 10-minute install.

In [2]:
# Colab ships torch preinstalled, so this runs before section 2's install.
# Elsewhere it may not be there yet -- say so rather than raising a bare
# ModuleNotFoundError.
try:
    import torch
except ModuleNotFoundError:
    raise SystemExit("torch is not installed in this kernel. On Colab it is "
                     "preinstalled; elsewhere run section 2's install first, "
                     "then re-run this cell.")

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device. Runtime -> Change runtime type -> T4 GPU -> Save, "
        "then re-run from the top. (config1-gemma-cpu/ is the CPU-only config; "
        "this one is GPU by definition.)")

GPU_NAME = torch.cuda.get_device_name(0)
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print("GPU: %s | %.1f GB | bf16 supported: %s"
      % (GPU_NAME, GPU_GB, torch.cuda.is_bf16_supported()))
print("\nRecord this line for the paper -- 'hardware' is one of the three things "
      "PLAN.md asks to record per program, and a free T4 reports bf16 supported: "
      "False, so a run that says True was not on a free T4.")

GPU: NVIDIA A100-SXM4-40GB | 42.4 GB | bf16 supported: True

Record this line for the paper -- 'hardware' is one of the three things PLAN.md asks to record per program, and a free T4 reports bf16 supported: False, so a run that says True was not on a free T4.


## 2. Install dependencies

`trl>=0.24` is a hard floor, not just a version bump -- `qlora_finetune.py`
relies on the current `trl` API (`SFTConfig` + `assistant_only_loss` over a
`"messages"`-shaped dataset). See `requirements.txt` for why older `trl`
won't work at all on this Python.

In [3]:
!pip -q install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 128.6 MB/s eta 0:00:00


## 3. Generate training data

Synthetic corpus from `corpus_gen.py`, with a disjoint id range and seed
from `eval-programs/` (which reserves ids 900-919). This step also runs
`check_no_leakage.py` automatically and refuses to proceed on any overlap
with `../eval-programs/` -- on program name *and* on exact source-text
hash. The "no leakage found" line below is that check passing, not just a
status message.

It writes two jsonl files:

- `../data/train.jsonl` -- what `--train` reads.
- `../data/train_holdout.jsonl` -- 20 examples split OFF the training set
  for `--eval`, so the eval loss printed at each epoch is measured on
  examples the optimizer never saw. This is a *training* diagnostic only.
  The scoring set is always `../eval-programs/`, which neither file
  touches.

Note what the eval set is **not** measuring: since 2026-09-21 the 20
`eval-programs/` files are hand-rewritten rather than `corpus_gen.py`
output (see `../eval-programs/README.md`), so they use different macros,
parameter names and join idioms than this training corpus. Config2 is
therefore being tested out-of-template, not just out-of-sample -- which
is the harder and more honest test, and worth saying in the writeup.

In [4]:
!python3 generate_training_corpus.py --n 600 --out-dir ../data

wrote 600 .sas programs to ../data/train_sas
wrote 600 gold JSON files to ../data/train_gold
wrote combined jsonl to ../data/train.jsonl
split 580 train / 20 holdout (holdout removed from train.jsonl)
no leakage found: 20 eval programs, training set clean.
no leakage found: 20 eval programs, training set clean.

training corpus ready at ../data/train.jsonl (+ ../data/train_holdout.jsonl ) -- verified clean against eval-programs/gold


## 4. Fine-tune

`google/gemma-3-4b-it` is gated on Hugging Face -- accept the license on
the model's page while logged in, then run the login cell below (or set
the `HF_TOKEN` Colab secret) before training can download the weights.
This is the same base model config1 serves via Ollama, kept matched on
purpose so "config1 vs config2" measures fine-tuning's effect, not a
different-model comparison. **Not** `google/gemma-4-E2B-it`: that name
reads like a small model but is Google's Gemma 3n E2B, a multimodal
checkpoint (vision + audio encoders, 10+ GB) that OOMs a free T4 inside
`prepare_model_for_kbit_training`, because those encoders mostly escape
4-bit quantization. See this folder's `SETUP.md`.

QLoRA (rank 32, attention + MLP target modules) on 600 synthetic examples,
2 epochs, `--maxlen 4096`, loss on the assistant turn only. Saves the
adapter to `../adapters/sasdoc-lora` once training finishes
(`trainer.model.save_pretrained` + tokenizer + a `run_config.json` of the
CLI args) -- no separate save step needed. `save_strategy="epoch"` also
drops a `Trainer` checkpoint into
`../adapters/sasdoc-lora/checkpoint-<step>/` at each epoch boundary.

**Runtime.** On a free T4 with `gemma-3-1b-it`, ~50s/optimizer-step was
observed, 150 steps total (600 examples / effective batch 8 / 2 epochs) --
budget **1.5-2.5 hours**, not the 30-90 min some earlier estimates
assumed. `gemma-3-4b-it` (this notebook's default) has been trained once
with these settings -- `../adapters/sasdoc-lora/run_config.json` records
`bf16=true`, which a free T4 does not support, so that run was on a bigger
GPU and its step time is not a T4 number. **No T4 end-to-end timing for
the 4B exists yet; record yours.**

**`--maxlen 4096` is not optional.** One example here is the schema block
+ a SAS program + its indented gold JSON, which measures ~2400-2800
tokens. At `--maxlen 2048` nothing fits: the 2026-09-22 run dropped
579/580 training and all 20 holdout examples and died with a bare
`StopIteration` from inside trl's `_prepare_dataset` on the empty eval
set. `qlora_finetune.py` now prints the measured token p50/p90/p99/max
per split and refuses to start when more than 20% of the corpus does not
fit, so that cannot come back as a silent one-example training run.

A `WARNING: this trl's SFTConfig does not accept: <field>` line is
expected and harmless: `requirements.txt` pins only a floor (`trl>=0.24`)
and `SFTConfig`'s field set drifts across releases, so the script filters
its kwargs against the resolved `trl` rather than crashing. `warmup_ratio`
is the exception -- if that spelling is gone the script converts it to an
equivalent `warmup_steps` instead of quietly training with no warmup.

In [5]:
from huggingface_hub import login
login()

In [ ]:
import time
t0 = time.time()
!python3 qlora_finetune.py --train ../data/train.jsonl \
    --eval ../data/train_holdout.jsonl \
    --model google/gemma-3-4b-it --maxlen 4096 --epochs 2 \
    --out ../adapters/sasdoc-lora
TRAIN_MINUTES = (time.time() - t0) / 60.0
print("\ntraining wall clock: %.1f min on %s" % (TRAIN_MINUTES, GPU_NAME))
print("This is a ONE-TIME cost. It does NOT belong in the results table's "
      "'cost per program' column, which is per-program inference -- report it "
      "separately (PLAN.md section 3, 'Also record').")

### Verify & persist the adapter

Still part of step 4: confirm the save worked, then get the adapter off
Colab's disk before the runtime recycles -- local storage does not survive
that. The adapter is ~130 MB (LoRA weights only, not the base model),
which is over GitHub's 100 MB limit, so it is gitignored: keep it in
Drive, download it, or push it to the HF Hub
(`qlora_finetune.py --push <repo-id>`).

In [ ]:
!ls -la ../adapters/sasdoc-lora

In [ ]:
if IN_COLAB:
    from google.colab import files
    import shutil
    shutil.make_archive("/content/sasdoc-lora", "zip", "../adapters/sasdoc-lora")
    files.download("/content/sasdoc-lora.zip")
else:
    print("not on Colab -- ../adapters/sasdoc-lora is already on local disk")

In [6]:
# RESUMING after a runtime recycle? Skip sections 3-4 and restore instead:
# upload the sasdoc-lora.zip you downloaded above, then run this cell.
# (Sections 5+ need only the adapter and the base model, not the training
# data.) Leave it commented out on a fresh run.
from google.colab import files
import shutil, os
up = files.upload()                      # pick sasdoc-lora.zip
os.makedirs("../adapters/sasdoc-lora", exist_ok=True)
shutil.unpack_archive(next(iter(up)), "../adapters/sasdoc-lora")
print(os.listdir("../adapters/sasdoc-lora"))

Saving sasdoc-lora.zip to sasdoc-lora.zip
['run_config.json', 'adapter_config.json', 'checkpoint-73', 'README.md', 'checkpoint-146', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja']


## 5. Generate predictions for scoring

Run **twice** on the identical 20 held-out `eval-programs/` -- once with
the adapter, once without -- so base-vs-tuned isn't confounded with a
different input set. Greedy decoding, so this is a fair comparison rather
than a sampling artifact.

Each run records per-program wall-clock generation time, the actual GPU
name read from `torch.cuda.get_device_properties`, and whether generation
hit `--max-new` into its `.meta.json` sidecar. `--gpu-usd-per-hour`
defaults to 0.0, which is correct on Colab's free tier and is what makes
the plan's air-gap sentence ("...at Y cost...") true; set it to what you
actually pay on a paid runtime and the results table's cost column
becomes a measured number instead of a hardcoded zero. Either way this is
inference cost only -- the one-time training cost above is separate.

In [7]:
!python3 infer.py --out ../results/preds/config2-base --gpu-usd-per-hour 0.0

GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
config.json: 100% 855/855 [00:00<00:00, 3.71MB/s]
tokenizer_config.json: 100% 1.16M/1.16M [00:00<00:00, 42.9MB/s]

tokenizer.json: downloading bytes:  25% 8.48M/33.4M [00:00<00:01, 13.6MB/s]
tokenizer.json: downloading bytes: 100% 8.83M/8.83M [00:00<00:00, 13.4MB/s,  866kB/s  ]
tokenizer.json: reconstructing file: 100% 33.4M/33.4M [00:00<00:00, 50.5MB/s, 3.28MB/s  ]
added_tokens.json: 100% 35.0/35.0 [00:00<00:00, 154kB/s]
special_tokens_map.json: 100% 662/662 [00:00<00:00, 2.99MB/s]
model.safetensors.index.json: 100% 90.6k/90.6k [00:00<00:00, 128MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/8.60G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):  48% 4.16G/8.60G [00:12<00:14, 301MB/s,  385MB/s  ]
Reconstructing (incomplete total...):  73% 6.24G/8.60G [00:17<00:06, 351MB/s,  331MB/s  ]
Reconstructing (incomp

In [8]:
!python3 infer.py --adapter ../adapters/sasdoc-lora \
    --out ../results/preds/config2-tuned --gpu-usd-per-hour 0.0

GPU: NVIDIA A100-SXM4-40GB (42.4 GB)
Loading weights: 100% 883/883 [00:02<00:00, 333.57it/s]
done prog900_estab (180.1s, 1164 tok)
done prog901_hhold (223.1s, 1425 tok)
done prog902_estcost (228.5s, 1466 tok)
done prog903_estcost (227.1s, 1455 tok)
done prog904_hhold (240.5s, 1547 tok)
done prog905_estcost (228.1s, 1455 tok)
done prog906_estcost (276.5s, 1771 tok)
done prog907_estcost (240.0s, 1535 tok)
done prog908_estcost (212.1s, 1356 tok)
done prog909_hhold (200.1s, 1287 tok)
done prog910_estab (163.0s, 1054 tok)
done prog911_estab (195.1s, 1257 tok)
done prog912_estab (204.0s, 1306 tok)
done prog913_estab (244.8s, 1592 tok)
done prog914_hhold (184.9s, 1201 tok)
done prog915_estab (205.0s, 1331 tok)
done prog916_estab (190.6s, 1241 tok)
done prog917_hhold (230.1s, 1495 tok)
done prog918_hhold (184.7s, 1199 tok)
done prog919_estab (181.9s, 1172 tok)

20/20 produced parseable JSON (0 needed the retry, 0 salvaged)


## 5b. Score the predictions

`results/run_eval.py` wraps `score.py` (schema validity, variable/macro/IO
precision-recall-F1, hallucination rate -- see its docstring) into the
`.scores.jsonl` files the plan's results table reads, and stamps a
`.provenance.json` sidecar recording the exact eval corpus scored. The
table marks a row **STALE** rather than printing numbers that no longer
refer to the corpus on disk; see
`../results/outputs/stale-2026-09-17/README.md` for the run that made
that necessary.

`--run-judge` is deliberately omitted: the default judge is a local Ollama
model, which Colab doesn't have. Run the judge later on a box that does --
the table prints `not run` for Description score rather than inventing
one.

In [13]:
!python3 ../results/run_eval.py --config config2-base \
    --pred-dir ../results/preds/config2-base \
    --out-prefix ../results/outputs/config2-base

[config2-base] wrote ../results/outputs/config2-base.scores.jsonl (20 programs)
[config2-base] wrote ../results/outputs/config2-base.provenance.json (corpus gold_digest=f27fb0130faf)


In [14]:
!python3 ../results/run_eval.py --config config2-tuned \
    --pred-dir ../results/preds/config2-tuned \
    --out-prefix ../results/outputs/config2-tuned

[config2-tuned] wrote ../results/outputs/config2-tuned.scores.jsonl (20 programs)
[config2-tuned] wrote ../results/outputs/config2-tuned.provenance.json (corpus gold_digest=f27fb0130faf)


### The results table

One `--table` call prints one header, so pass every row at once rather
than concatenating separate invocations. `../results/rows.example.json`
lists all the configs; rows with no scores yet print
`(no scores found)` instead of failing, so this is also how you check
where the comparison currently stands.

In [15]:
!python3 ../results/run_eval.py --table --rows ../results/rows.example.json

eval corpus on disk: 20 gold / 20 programs (gold_digest f27fb0130faf)

| Config | Schema validity | Variable F1 | Macro F1 | I/O F1 | Hallucination rate | Description score | Time/program | Cost/program |
|---|---|---|---|---|---|---|---|---|
| Config 1: Gemma CPU (ollama, gemma3:1b) | (no scores found) | | | | | | | |
| Config 1: Gemma CPU (hf, gemma-3-4b-it) | (no scores found) | | | | | | | |
| Config 2: base (no adapter) | 0.45 [0.20, 0.65] | 0.01 [0.00, 0.02] | 0.18 [0.04, 0.34] | 0.10 [0.03, 0.18] | 0.57 [0.37, 0.81] | not run | 88.3s | $0.00 (local) |
| Config 2: QLoRA-tuned | 1.00 [1.00, 1.00] | 0.92 [0.90, 0.94] | 1.00 [1.00, 1.00] | 1.00 [1.00, 1.00] | 0.00 [0.00, 0.00] | not run | 212.0s | $0.00 (local) |
| Config 3: Claude + skills | (no scores found) | | | | | | | |

Config 2: base (no adapter): 20 program(s), [1] run(s) each; 20/20 scored outputs parsed as JSON, 9/20 passed schema.validate().
    NOTE: 1 run only. PLAN.md asks for 3 runs per config to separate model varia

### Get the results off Colab

The adapter was saved above; these are the run outputs. Both are gone
when the runtime recycles.

In [16]:
if IN_COLAB:
    from google.colab import files
    import shutil
    for src, name in (("../results/preds", "config2-preds"),
                      ("../results/outputs", "config2-outputs")):
        shutil.make_archive("/content/" + name, "zip", src)
        files.download("/content/%s.zip" % name)
else:
    print("not on Colab -- outputs are already on local disk")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. (Optional) Ground truth from SAS OnDemand for Academics (ODA)

Not needed for the base-vs-tuned comparison above. What it buys you is a
**fairer hallucination rate**: `oda_harvest.py` actually runs each eval
program on ODA and reads back `dictionary.columns`, so the scorer can
count a column SAS really materialized as a valid source even when
`extract.py`'s static regex scan missed it. Per `PLAN.md` section 3 that
counts as a valid source, and section 5b above scores without it -- so a
name config2 got *right* can currently be flagged as invented.

Everything in this section runs on Colab. The four things that differ from
the local box:

1. **Credentials** come from Colab Secrets (key icon, left sidebar:
   `ODA_USER`, `ODA_PASS`), written into the session's `~/.authinfo` at
   chmod 600. Never paste a password into a cell that gets saved.
2. **Java**: Colab has no system Java and this repo's portable `jre/` is
   gitignored (136 MB), so it is not in the clone -- `apt-get install
   default-jdk` below. Colab runs as root, so no sudo is needed.
   `sascfg_personal.py` in this folder already points `java` at whatever
   is on `PATH` for exactly this reason.
3. **Region**: `sascfg_personal.py`'s `iomhost` list is filled in for
   US-region/usw2. If your ODA account is Europe or Asia Pacific, edit it
   -- config1's `SETUP.md` lists those host names.
4. **`--out` is a directory**, not a file: `oda_harvest.py` writes one
   `<program>.json` and one `<program>.txt` per program, and
   `run_eval.py --extra-source-dir` reads that directory.

**Licence note:** ODA is for academic/non-commercial use. These are
synthetic programs, which is fine -- check current terms before putting
anything work-adjacent on it.

In [18]:
import getpass

authinfo = os.path.expanduser("~/.authinfo")
if os.path.exists(authinfo):
    print("~/.authinfo found -- leaving it alone")
else:
    user = password = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            user, password = userdata.get("ODA_USER"), userdata.get("ODA_PASS")
        except Exception:
            pass
    user = user or input("ODA email: ")
    password = password or getpass.getpass("ODA password: ")
    with open(authinfo, "w") as fh:
        fh.write("oda user %s password %s\n" % (user, password))
    os.chmod(authinfo, 0o600)
    del password
    print("wrote", authinfo, "(chmod 600)")

wrote /root/.authinfo (chmod 600)


In [19]:
import shutil
if IN_COLAB:
    !apt-get -qq install default-jdk > /dev/null
!pip -q install saspy pandas
print("java on PATH:", shutil.which("java") or "STILL MISSING -- saspy cannot connect")

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-21/openjdk-21-jre_21.0.12%2b8-1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.92.22 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-21/openjdk-21-jdk_21.0.12%2b8-1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.92.22 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?
java on PATH: /usr/bin/java


### Preflight: what breaks this harvest on Colab

Two failures worth catching before a SAS connection attempt spends a
minute and then dies in a Java traceback. The cell below checks both and
repairs the first.

1. **A hand-built `classpath` in `sascfg_personal.py`.** Earlier commits of
   this repo set one, listing only `saspy/java/iomclient/*.jar` -- which
   omits the CORBA back-port saspy ships in `java/thirdparty/`. SAS's IOM
   protocol needs `org.omg.CORBA.*`; the JDK carried it through Java 8 and
   dropped it in JEP 320, so on the JDK `apt-get` just installed the Java
   helper dies with

   ```
   Error: Unable to initialize main class pyiom.saspy2j
   Caused by: java.lang.NoClassDefFoundError: org/omg/CORBA/COMM_FAILURE
   ...
   SAS Connection failed. No connection established.
   ```

   and saspy reports `SAS process has terminated unexpectedly`. The fix is
   to not set `classpath` at all -- saspy's own default includes the
   back-port (and the current jar names; that old list also named
   `log4j.jar` and `sas.rutil.jar`, which current saspy builds don't
   ship). The cell strips the key if this clone still has it, so an older
   checkout repairs itself instead of failing.

2. **Whether the back-port jars are actually in this saspy build.** If
   `java/thirdparty/` is empty, no classpath can save it -- install a
   newer `saspy`.

If you see that CORBA error anyway, you are running an older clone than
the one that fixed it: `git -C /content/sas-llm-paper pull`, restart the
runtime, and re-run.

In [20]:
import re
import saspy

CFG = "sascfg_personal.py"
cfg_text = open(CFG).read()
if re.search(r'^\s*"classpath"\s*:', cfg_text, re.M):
    open(CFG, "w").write(re.sub(r'^\s*"classpath"\s*:.*\n', "", cfg_text, flags=re.M))
    print("PATCHED %s: removed a hand-built classpath (stale clone) -- saspy's own "
          "default, which carries the CORBA back-port, is used instead." % CFG)
else:
    print("%s: no hand-built classpath -- good" % CFG)

thirdparty = os.path.join(os.path.dirname(saspy.__file__), "java", "thirdparty")
jars = sorted(f for f in os.listdir(thirdparty) if f.endswith(".jar")) \
    if os.path.isdir(thirdparty) else []
print("saspy %s CORBA back-port jars: %s" % (saspy.__version__, ", ".join(jars) or
      "NONE FOUND -- pip install -U saspy"))
print("java on PATH:", shutil.which("java") or "MISSING -- re-run the apt-get cell")

sascfg_personal.py: no hand-built classpath -- good
saspy 5.109.3 CORBA back-port jars: glassfish-corba-internal-api.jar, glassfish-corba-omgapi.jar, glassfish-corba-orb.jar, pfl-basic.jar, pfl-tf.jar
java on PATH: /usr/bin/java


`oda_harvest.py` already passes this folder's `sascfg_personal.py` as
`--cfgfile`, so there is no need to copy it next to the saspy install
(earlier drafts of `SETUP.md` said to; it was redundant).

The harvest submits each program to a real SAS session. A program that
partially fails still gets its materialized datasets harvested, and any
`ERROR` lines are printed rather than swallowed -- read them, because a
program that errored out contributes less ground truth than one that
didn't.

In [21]:
!python3 oda_harvest.py --sas-dir ../eval-programs/programs --out ../data/oda_metadata

Using SAS Config named: oda
SAS Connection established. Subprocess id is 49532

Access Method         = IOM
SAS Config name       = oda
SAS Config file       = /content/sas-llm-paper/config2-qlora-gpu/sascfg_personal.py
WORK Path             = /saswork/SAS_work696B00018D76_odaws01-usw2.oda.sas.com/SAS_workFFA800018D76_odaws01-usw2.oda.sas.com/
SAS Version           = 9.04.01M8P02222023
SASPy Version         = 5.109.3
Teach me SAS          = False
Batch                 = False
Results               = Pandas
SAS Session Encoding  = utf-8
Python Encoding value = utf-8
SAS process Pid value = 101750
SASsession started    = Wed Sep 23 01:56:12 2026


harvested prog900_estab.sas -> 39 column(s)
harvested prog901_hhold.sas -> 52 column(s)
harvested prog902_estcost.sas -> 24 column(s)
harvested prog903_estcost.sas -> 44 column(s)
harvested prog904_hhold.sas -> 45 column(s)
harvested prog905_estcost.sas -> 39 column(s)
harvested prog906_estcost.sas -> 28 column(s)
harvested prog907_estcost.sas 

### Re-score against the harvested ground truth

This is the payoff for section 6, and the step that was missing before:
without it the harvest sits in `../data/oda_metadata/` and changes
nothing.

Scored into a **separate** `-gt` prefix on purpose. A hallucination rate
measured with real SAS metadata as an allowed source is not comparable to
one measured without it, so the two must not overwrite each other or share
a row. `run_eval.py` records which was which in the provenance sidecar's
`notes` and per-row `used_extra_source`, and the plan asks for the tool-
access asymmetry to be stated explicitly rather than folded into the
headline number.

In [22]:
!python3 ../results/run_eval.py --config config2-tuned-gt \
    --pred-dir ../results/preds/config2-tuned \
    --extra-source-dir ../data/oda_metadata \
    --out-prefix ../results/outputs/config2-tuned-gt

[config2-tuned-gt] wrote ../results/outputs/config2-tuned-gt.scores.jsonl (20 programs)
[config2-tuned-gt] wrote ../results/outputs/config2-tuned-gt.provenance.json (corpus gold_digest=f27fb0130faf)
[config2-tuned-gt] hallucination check also allowed 20/20 programs' real SAS metadata from ../data/oda_metadata as a valid source


In [23]:
!python3 ../results/run_eval.py --table \
    --scores ../results/outputs/config2-tuned.scores.jsonl \
    --meta-dir ../results/preds/config2-tuned \
    --label "Config 2: QLoRA-tuned (static scan only)"
!python3 ../results/run_eval.py --table \
    --scores ../results/outputs/config2-tuned-gt.scores.jsonl \
    --meta-dir ../results/preds/config2-tuned \
    --label "Config 2: QLoRA-tuned (+ real SAS metadata)"

eval corpus on disk: 20 gold / 20 programs (gold_digest f27fb0130faf)

| Config | Schema validity | Variable F1 | Macro F1 | I/O F1 | Hallucination rate | Description score | Time/program | Cost/program |
|---|---|---|---|---|---|---|---|---|
| Config 2: QLoRA-tuned (static scan only) | 1.00 [1.00, 1.00] | 0.92 [0.90, 0.94] | 1.00 [1.00, 1.00] | 1.00 [1.00, 1.00] | 0.00 [0.00, 0.00] | not run | 212.0s | $0.00 (local) |

Config 2: QLoRA-tuned (static scan only): 20 program(s), [1] run(s) each; 20/20 scored outputs parsed as JSON, 20/20 passed schema.validate().
    NOTE: 1 run only. PLAN.md asks for 3 runs per config to separate model variance from program difficulty -- the CI below reflects program difficulty alone.

Hallucination rate convention: a program that produced no parseable output is scored 1.00 (worst case), not skipped -- see score.py's docstring. Check the coverage line above before reading that column as 'the model invented things'.
eval corpus on disk: 20 gold / 20 progr

## What to report honestly from this config

- **Training cost is not inference cost.** The results table's
  "cost/program" column is per-program generation. Report the one-time
  training wall clock and GPU separately (section 4 prints it).
- **The GPU is recorded, not described.** `.meta.json` now carries the
  real device name; a run claiming `bf16 supported: True` was not on a
  free T4.
- **Base vs tuned is the internal comparison.** Config2-vs-config1 also
  changes the serving stack (bitsandbytes 4-bit on GPU vs Ollama GGUF on
  CPU, and 4B vs 1B by default), so `config2-base` is the cleaner control
  for "what did fine-tuning do".
- **Out-of-template, not just out-of-sample.** The eval programs are
  hand-written and deliberately unlike `corpus_gen.py`'s templates; a
  QLoRA adapter trained purely on templates may transfer less well than a
  same-template holdout would suggest. That gap is a finding, not a bug.
- **Truncation is a failure mode, not a wrong answer.** `.meta.json`'s
  `truncated` flag says generation hit `--max-new` mid-JSON. Those score
  as schema-invalid; check how many before reading the schema-validity
  column as a statement about the model's formatting ability.
- **`--gpu-usd-per-hour 0.0`** is a claim about the free tier. If you ran
  on a paid runtime and left it at 0, the cost column is wrong.